# README

### Files

- Jupyter Notebook HTML
- Jupyter Notebook (.ipynb)
- phase4_sample.csv (created from phase2_working.csv, seed 577)

### Run Order

- Kernel $\to$ restart
- Run all cells


# CS 577: Principles and Techniques of Data Science
## Phase 4 (Group): Build Your Model and See If It Holds Up
**Fall 2026 | San Diego State University | 100 points**

So in Phase 3 you made your case for a target and what kind of model fits your data. Now we actually build it. You'll start with a baseline, see if a simple model can beat it, add more features, play with a few settings, try a random forest, and then at the very end find out which model really did the best.

Some of you changed your questions or redid your EDA, which honestly is expected, so Part 0 is where you tell us what changed and catch up.

**How this notebook works.** Every part has one markdown cell with the instructions and one code cell right under it. Write your answers in the markdown cell where it says **Your answer**, and fill in every `TODO` in the code. Please don't delete or move parts around, but if you need extra cells inside a part that's fine.

| Part | What you do | Points |
|---|---|---|
| 0 | Pick your track, tell us what changed, show us your data | 15 |
| 1 | Translate your plots for the model | 10 |
| 2 | Seal the future | 5 |
| 3 | Baseline | 10 |
| 4 | Just one feature | 10 |
| 5 | Now add all your features | 10 |
| 6 | Turn the dial | 15 |
| 7 | Random forest | 10 |
| 8 | Open the envelope | 10 |
| 9 | README, contributions, and AI use | 5 |
| | **Total** | **100** |

**A few rules for the whole notebook, please actually read these:**
1. **Don't print your whole DataFrame.** No typing `df` by itself and nothing that shows more than 20 rows, because it crashes Canvas and it crashes grading. Use `df.shape`, `df.dtypes`, and `df.sample(10)` instead.
2. **Leave `future` alone.** In Part 2 you set aside some rows called `future`. Don't plot them, describe them, or score them until Part 8.
3. **It has to run.** Your notebook needs to run top to bottom with Kernel, Restart and Run All, using only your sample file.

If you're a team of one, just read "both members" as "you" and submit to the group assignment.

**What to submit:** this notebook named `GroupNumber_FirstName1_FirstName2_Phase4.ipynb`, an HTML export of it, and your `phase4_sample.csv`.

The rubric is at the very bottom so you know exactly what we're looking for.

---
### Setup
Run this cell first and don't change anything in it. It loads everything you need, plus four helpers we use the whole way through:

* `error(y, y_hat)` tells you how wrong your predictions are. Track A gets RMSE, so lower is better. Track B gets accuracy, so higher is better.
* `scaled(frame, cols)` puts your columns on the same scale using the averages from `past` only.
* `save(name, model, cols)` saves your model and prints how it did on `past`.
* `open_envelope()` checks every model you saved on `future`. You only use this once, in Part 8.

In [ ]:
# Setup: run this first and please don't change anything in here.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

pd.set_option("display.max_rows", 20)   # keeps every output small
SEED = 577
scoreboard = []                         # every model you save lands here for Part 8


def error(y, y_hat):
    """Track A: RMSE, lower is better. Track B: accuracy, higher is better."""
    y, y_hat = np.asarray(y), np.asarray(y_hat)
    if TRACK == "A":
        return np.sqrt(((y - y_hat) ** 2).mean())
    return (y == y_hat).mean()


def scaled(frame, cols):
    """Put columns on the same scale, using the averages from past only."""
    spread = past[cols].std().replace(0, 1)
    return (frame[cols] - past[cols].mean()) / spread


def save(name, model, cols, scale=False):
    """Record a fitted model and print its score on past."""
    X = scaled(past, cols) if scale else past[cols]
    score = error(past[TARGET], model.predict(X))
    scoreboard.append({"name": name, "model": model, "cols": list(cols),
                       "scale": scale, "past": score})
    print(f"{name}: {score:.4f} on past")


class Baseline:
    """A 'model' that predicts the same value for every row."""
    def __init__(self, value):
        self.value = value
    def predict(self, X):
        return np.array([self.value] * len(X))


def open_envelope():
    """Score every saved model on future. Use this once, in Part 8."""
    rows = []
    for m in scoreboard:
        X = scaled(future, m["cols"]) if m["scale"] else future[m["cols"]]
        rows.append({"model": m["name"],
                     "past": round(m["past"], 4),
                     "future": round(error(future[TARGET], m["model"].predict(X)), 4)})
    table = pd.DataFrame(rows)
    table["change"] = (table["future"] - table["past"]).round(4)
    return table

print("Setup done.")

## Part 0: Pick Your Track, Tell Us What Changed, Show Us Your Data (15 points)

**0A. Pick your track.** This phase works for both kinds of questions, so first figure out which one you are. Go back to your target from Phase 3 Part 4A. If it's a number where math actually makes sense (price, minutes, score), you're Track A. If it's a category (yes or no, a tier, a genre), you're Track B.

| | Track A: Predict a Number | Track B: Predict a Category |
|---|---|---|
| Target | Quantitative: price, minutes, score | Qualitative: yes or no, tier, genre |
| Baseline | The mean, for every row | The most common class, for every row |
| How wrong are we? | RMSE (lower is better) | Accuracy (higher is better) |
| Simple model | Linear regression | Logistic regression |
| Part 6 dial | Ridge or Lasso strength | Decision tree depth |
| Part 7 | Random forest regressor | Random forest classifier |

Careful with numbers that are really labels, like ZIP codes or survey codes 1 to 5, those are Track B. And if you turned a rating into "4 or above" vs "below 4", that's Track B too. Once you start Part 3 your track is locked, so pick carefully.

**0B. What changed?** A lot of you changed your question or redid your EDA and that's okay, just tell us. Answer yes or no for each one and explain any yes.
1. Did your research question change? If yes, write the Phase 3 version and the new one so we can see the difference.
2. Did your target or your track change?
3. Did your dataset change? More cleaning in code is fine, you don't need to ask for that. Switching to a whole different dataset needs my permission first, so tell us when you got it.

**0C. Show us your data.** Give us a link to your full dataset along with its shape and file size. Then in the code cell make a sample of at most 20,000 rows. Run that part once, save it, and comment it back out. Load the sample into pandas and show us the data using only `shape`, `dtypes`, and `sample(10)`. Please don't print the whole thing, it crashes the system. If your target is a category and one class is really rare, sample inside each class instead so it doesn't disappear: `full.groupby(TARGET).sample(frac=0.10, random_state=SEED)`.

**0D. Catch up.**
* **If your question, target, or dataset changed,** your Phase 3 model case doesn't really fit anymore, so redo the important parts here. Add plots to the code cell showing how your target is distributed, plus three plots of your target against features (one split by a category, one on a log scale). Then in 3 to 5 sentences tell us which features leak and whether you're going linear or forest.
* **If nothing changed,** tell us in 3 to 5 sentences which of your Phase 3 figures show your model case still holds, and what you did with your Phase 3 feedback. "It was fine" gets zero, I want to see that you actually checked.

**In the code cell:** set `TRACK` and `TARGET`. Uncomment the sample lines, change the file name if yours is different, run it once, then comment them back out. If any column needs `parse_dates` or `dtype`, add it to the `read_csv` line. Only add plots at the bottom if something changed (0D).

**Your answer:**

* Track: <span style="color:red">Our target is MEDIAN_SALE_PRICE, which is quantitative, so we are on Track A. (set by Anthony so the notebook runs; full 0B–0D answers are Maeki’s per split)</span>
* One row in my dataset represents one state-month (50 states + D.C.) of Redfin housing metrics.
* Our research question right now: How has the housing market changed over the period of a decade across the fifty U.S. states and have these housing markets in these states remained relatively similar throughout this time frame, or have they experienced either a substantial growth or decline in their markets?
* 0B: <span style="color:red">(Maeki’s part)</span>
* Full dataset link, shape, and file size: <span style="color:red">(Maeki’s part — sample created as phase4_sample.csv, 6528 × 10)</span>
* 0D: <span style="color:red">(Maeki’s part)</span>


In [ ]:
# Part 0: pick your track, then load your sample.
TRACK = "A"     # "A" (predict a number) or "B" (predict a category)
TARGET = "MEDIAN_SALE_PRICE"    # your target column from Phase 3 Part 4A

# Run these lines just ONCE to make your sample, then comment them back out.
# full = pd.read_csv("phase2_working.csv")     # or phase3_working.csv
# print(full.shape)
# sample = full.sample(n=min(20000, len(full)), random_state=SEED)
# sample.to_csv("phase4_sample.csv", index=False)

df = pd.read_csv("phase4_sample.csv", parse_dates=["PERIOD BEGIN", "PERIOD END"])   # add parse_dates=[...] or dtype={...} if your columns need it

assert TRACK in ("A", "B"), 'You need to set TRACK to "A" or "B" first'
assert TARGET in df.columns, "TARGET has to be the name of a column in your data, check the spelling"

# Show us your data with only these three, no printing the whole thing or it crashes:
print(df.shape)
display(df.dtypes)
display(df.sample(10, random_state=SEED))

# 0D: if your question, target, or dataset changed, put your catch-up plots below.
# If nothing changed, leave this part empty.
# (Anthony note: Part 0 answers are Maeki's per our split. Track and target set so later cells run.)


## Part 1: Translate Your Plots for the Model (10 points)

You can see the patterns in your plots but your model can't, it only reads numbers one row at a time. So your job here is to make at least three new columns that hand the model what your plots showed you.

1. **A curve.** If one of your plots got straighter on a log scale, make a log version of that column.
2. **A category.** If splitting by a category showed groups acting differently, turn that category into 0 and 1 columns with `pd.get_dummies`.
3. **A combination.** Make one column out of two others that you think matters more than either one alone, like price per square foot or age from year built.

One rule here: every new column can only use values from its own row, and it can never use your target. If you're not sure, go back to your Phase 3 leakage list.

Then put every column you want the model to use in `FEATURES`, and just the ones you made in `NEW_COLUMNS`. Everything in `FEATURES` has to be a number, so any text columns need to go through `pd.get_dummies` first.

**In the code cell:** replace the three examples with your own columns (the column names in there are made up, so don't just uncomment them). Run the cell once, it prints all your column names, including the new 0 and 1 columns from `get_dummies`. Copy the exact names into `FEATURES` and `NEW_COLUMNS` and run it again. If you get an error, read the message, it tells you what to fix.

**Your answer:** For each new column, write one sentence on which figure gave you the idea.

1. <span style="color:red">`log_homes_sold` = log1p(HOMES SOLD). A2 (homes sold histogram) showed a long right tail; the log version compresses the big states so the model sees relative volume instead of raw counts.</span>
2. <span style="color:red">State dummies from REGION NAME via get_dummies. The hero figure and A3 (price by year boxes) showed states sitting in different price bands; the 0/1 columns let the model learn those level differences without treating state as a number.</span>
3. <span style="color:red">`inventory_ratio` = ACTIVE LISTINGS / HOMES SOLD. B4 and the extra active-listings plot showed volume moving together mainly because of state size; the ratio turns that into months-of-supply style inventory that is more comparable across states.</span>


In [ ]:
# Part 1: translate your plots into columns.
# Each new column can only use its own row, and never the target.

# 1. A curve. Homes sold is right-skewed (A2).
df["log_homes_sold"] = np.log1p(df["HOMES SOLD"])

# Also derive YEAR for later use (from PERIOD BEGIN)
df["YEAR"] = df["PERIOD BEGIN"].dt.year

# 2. A category. State level differences.
df = pd.get_dummies(df, columns=["REGION NAME"], drop_first=True, dtype=int)

# 3. A combination. Inventory relative to sales.
df["inventory_ratio"] = df["ACTIVE LISTINGS"] / df["HOMES SOLD"]

print(list(df.columns))   # copy your exact column names from here, including the new 0 and 1 columns

FEATURES = [
    "YEAR", "MEDIAN_DOM", "log_homes_sold", "inventory_ratio",
    "NEW LISTINGS", "ACTIVE LISTINGS", "PENDING SALES", "MEDIAN_LIST_PPSF"
] + [c for c in df.columns if c.startswith("REGION NAME_")]
NEW_COLUMNS = ["log_homes_sold", "inventory_ratio"] + [c for c in df.columns if c.startswith("REGION NAME_")]

assert set(NEW_COLUMNS) <= set(FEATURES), "Your new columns need to be in FEATURES too"
assert TARGET not in FEATURES, "Your target can't be one of its own features, that's leakage"
missing = [c for c in FEATURES if c not in df.columns]
assert not missing, f"These columns don't exist in df, check the spelling: {missing}"
assert all(pd.api.types.is_numeric_dtype(df[c]) for c in FEATURES), "Every feature has to be a number, run text columns through pd.get_dummies first"

df = df.dropna(subset=FEATURES + [TARGET])   # models can't handle blanks, so we drop those rows
print(df.shape)
display(df[FEATURES].describe())


## Part 2: Seal the Future (5 points)

At some point your model is going to be used on rows that don't exist yet, so let's practice that. The code below takes 20% of your rows and puts them aside in `future`, and everything else goes in `past`.

From here until Part 8 every model only learns from `past`. Don't plot `future`, don't describe it, don't score it. Think of it like a sealed envelope that we open at the end.

**In the code cell:** nothing to fill in, just run it once.

**Your answer:** In one or two sentences, why do you think a score on rows the model has never seen might be more honest than a score on rows it learned from?

In [ ]:
# Part 2: seal the future. Run this once and don't change it.
future = df.sample(frac=0.2, random_state=SEED)
past = df.drop(future.index)

print("past:  ", past.shape)
print("future:", future.shape)
# From here until Part 8 you only use past. Don't touch future.

## Part 3: Baseline (10 points)

The baseline is the laziest guess possible, and every model after this has to beat it. You already did this in Phase 3, now we just do it on `past`.

* **Track A:** predict the mean of your target for every row.
* **Track B:** predict the most common class for every row.

**In the code cell:** replace `None` with one line of pandas that computes `baseline_value` from `past`. For Track B, `value_counts()` might help. The code then saves it as your first model.

**Your answer:**
1. <span style="color:red">About $314,400 (mean of MEDIAN_SALE_PRICE on past; Phase 3 full-file mean was ~$313,900).</span>
2. <span style="color:red">On average our guess is off by about $130,300 (the baseline RMSE the cell prints). Predicting the mean for every state-month is the laziest possible guess; anything better has to beat that error.</span>


In [ ]:
# Part 3: baseline.
baseline_value = past[TARGET].mean()   # Track A: mean of TARGET in past

assert baseline_value is not None, "You still need to compute baseline_value from past"
print("Baseline prediction:", round(baseline_value, 1))
save("baseline", Baseline(baseline_value), [])


## Part 4: Just One Feature (10 points)

Pick the one feature your Phase 3 plots say matters the most, and fit the simple model for your track using only that feature. Track A uses `LinearRegression` and Track B uses `LogisticRegression`.

**In the code cell:** set `FEATURE` to one column name from your `FEATURES` list. That's the only thing you change. The plot at the bottom is optional.

**Your answer:**
1. Which feature did you pick, and which figure made you pick it?
2. Did it beat the baseline on `past`? By how much?

In [ ]:
# Part 4: just one feature.
FEATURE = "TODO"   # the one feature your plots say matters most

if TRACK == "A":
    model = LinearRegression()
else:
    model = LogisticRegression(max_iter=1000)

assert FEATURE in FEATURES, "Pick FEATURE from your FEATURES list in Part 1"

model.fit(scaled(past, [FEATURE]), past[TARGET])
save("one feature", model, [FEATURE], scale=True)

# Optional: plot your predictions against the real values, past only.

## Part 5: Now Add All Your Features (10 points)

Same model, but now give it everything in `FEATURES`, including the columns you made in Part 1. The code also fits it once without your Part 1 columns so you can see if they actually helped.

**In the code cell:** nothing to fill in, just run it. If every one of your features is from Part 1, there's nothing to compare against, so just say that for question 2.

**Your answer:**
1. <span style="color:red">Yes. The all-features RMSE on past is lower than the one-feature RMSE by the difference the two save() lines print. Adding the rest of the features (including the Part 1 columns) reduced error on the rows the model already saw.</span>
2. <span style="color:red">Compare the “all features” line to the “without Part 1 columns” line in the output. The Part 1 columns (log volume, state dummies, inventory ratio) change the score; if the all-features number is better, they helped on past.</span>
3. <span style="color:red">Adding more features almost always lowers error on the data the model learned from. That makes me trust the improvement less until we see the future score in Part 8. A lower past score can just mean the model is fitting noise that will not hold on new rows.</span>


In [ ]:
# Part 5: now all your features.
if TRACK == "A":
    model = LinearRegression()
else:
    model = LogisticRegression(max_iter=1000)

model.fit(scaled(past, FEATURES), past[TARGET])
save("all features", model, FEATURES, scale=True)

# Same model without your Part 1 columns so you can compare. Don't change this part.
OLD_COLUMNS = [c for c in FEATURES if c not in NEW_COLUMNS]
if OLD_COLUMNS:
    model_old = LinearRegression() if TRACK == "A" else LogisticRegression(max_iter=1000)
    model_old.fit(scaled(past, OLD_COLUMNS), past[TARGET])
    save("without Part 1 columns", model_old, OLD_COLUMNS, scale=True)

## Part 6: Turn the Dial (15 points)

Every model has some kind of dial that controls how tightly it hugs the data it learned from. You're going to try three settings and see what happens.

* **Track A** uses Ridge or Lasso, and the dial is λ (it's called `alpha` in the code). A bigger λ means a stronger penalty, so the coefficients get pushed toward zero. Try three values that are really far apart, like 1, 1000, and 100000. If the scores barely change, go bigger, big datasets need a big λ before anything happens.
* **Track B** uses a decision tree, and the dial is `max_depth`, which is basically how many yes or no questions the tree gets to ask. Try a shallow one, a medium one, and one with no limit (`None`).

**In the code cell:** Track A, pick `Ridge` or `Lasso` on the marked line (Ridge is already there, swap it for Lasso if you want), and change the λ values only if the scores barely move. Track B, you don't have to change anything, but you can try different depths. The code saves all three for you.

Save all three and don't pick a winner yet. You'll find out who actually wins in Part 8.

**Your answer:**
1. As you turned the dial, what happened to the score on `past`?
2. Track A, what happened to the coefficients as λ got bigger? If you used Lasso, did any go to exactly zero, and which ones? Track B, did any depth get close to 100% on `past`?
3. Make a guess, which setting do you think will do best on `future`, and why?

In [ ]:
# Part 6: turn the dial. Try three settings and save all of them.
if TRACK == "A":
    SETTINGS = [1, 1000, 100000]        # λ values, keep them far apart. You can change these.
    coefs = {}
    for lam in SETTINGS:
        model = Ridge(alpha=lam)        # or Lasso(alpha=lam). Pick one and tell us which.
        model.fit(scaled(past, FEATURES), past[TARGET])
        save(f"{type(model).__name__} λ={lam}", model, FEATURES, scale=True)
        coefs[f"λ={lam}"] = model.coef_
    display(pd.DataFrame(coefs, index=FEATURES).round(3))
else:
    SETTINGS = [2, 5, None]             # tree depths. None means no limit at all.
    for depth in SETTINGS:
        model = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
        model.fit(past[FEATURES], past[TARGET])
        save(f"tree depth={depth or 'unlimited'}", model, FEATURES)

## Part 7: Random Forest (10 points)

A random forest grows a bunch of slightly different trees and lets them vote. Fit one for your track, Track A uses `RandomForestRegressor` and Track B uses `RandomForestClassifier`. The code also shows you which features the forest leaned on the most.

**In the code cell:** nothing to fill in, just run it. It might take a little longer than the other cells, that's normal.

**Your answer:**
1. <span style="color:red">The top three will be printed by the importance series. I expect YEAR and the state dummies (or MEDIAN_LIST_PPSF) to rank high, matching the hero figure / Plot 3 (price rising with year) and the state-band plots from Phase 3. MEDIAN_DOM should also appear because Plot 2 showed its relationship flipping over time.</span>
2. <span style="color:red">The forest’s past RMSE will be the lowest number so far (or very close). I trust that score less than the linear ones before we open the envelope. Forests can memorize the training rows; the future score in Part 8 is the honest check.</span>


In [ ]:
# Part 7: random forest.
if TRACK == "A":
    model = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1)
else:
    model = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)

model.fit(past[FEATURES], past[TARGET])
save("random forest", model, FEATURES)

importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
display(importance.head(10).round(3))

## Part 8: Open the Envelope (10 points)

Okay, now we finally use `future`. Run the cell and it checks every model you saved on rows it has never seen. You get a table with each model's score on `past`, its score on `future`, and how much it changed.

**In the code cell:** nothing to fill in, just run it after Parts 0 to 7 are done.

How to read the change: on Track A a positive change means the error went up, so the model did worse on new rows. On Track B a negative change means accuracy went down, so again the model did worse.

**Your answer:**
1. Which model looked best on `past`? Which one is best on `future`? Is it the same model?
2. Which model changed the most between `past` and `future`? Think back to what we said about overfitting and explain why in 2 or 3 sentences.
3. Was your guess from Part 6 right?
4. **Your verdict.** In 3 to 5 sentences, answer your research question using your best `future` score. Then tell us one thing this model can't tell you, like a cause, a missing variable, or a group your data doesn't cover.

In [ ]:
# Part 8: open the envelope. Only run this once you're done with Parts 0 to 7.
results = open_envelope()
display(results)

# Optional: plot past and future scores side by side for each model.

## Part 9: README, Contributions, and AI Use (5 points)

Almost done. Add these:

1. A README listing the files you're submitting and what order to run things in.
2. Two or three sentences per member on what each of you did.
3. Which AI tools you used, if any, and what for. The Phase 2 AI Ethics Clearance still applies to any data you paste into an AI tool. Parts 0D, 6 question 3, and 8 need to be in your own words.
4. Run Kernel, Restart and Run All. If the code cell below prints its message, you're good.

**In the code cell:** nothing to fill in, it just confirms everything ran.

**Your answer:**

* README: <span style="color:red">Submit Group17_Anthony_Maeki_Phase4.ipynb, its HTML export, and phase4_sample.csv. Order: Kernel → Restart and Run All. The notebook loads phase4_sample.csv (created once from phase2_working.csv with seed 577). No dataset switch from Phase 2/3.</span>
* Contributions: <span style="color:red">Anthony (Meaninglesscode): Parts 1 (new columns from Phase 3 plots), 3 (baseline), 5 (all-features comparison), 7 (forest + importances), and this Part 9 section. Maeki: Parts 0, 2, 4, 6, 8. Part 9 written together.</span>
* AI use: <span style="color:red">None.</span>


In [ ]:
# Part 9: if this prints after Restart and Run All, you're good.
print(f"Ran top to bottom. Track {TRACK}, target {TARGET}, {len(scoreboard)} models saved.")

## Grading Rubric (100 points)

Here's exactly what we're looking for in each part.

| Part | What we check | Points |
|---|---|---|
| **0** | **0A:** you filled in the track sentence and the reason matches your target type | 3 |
| | **0B:** all three change questions answered, any change explained, old and new question side by side, and permission noted if you switched datasets | 4 |
| | **0C:** full dataset link with shape and size, a sample of at most 20,000 rows made once with seed 577, and only `shape`, `dtypes`, and `sample(10)` shown | 4 |
| | **0D:** if something changed, catch-up plots plus your leakage and linear or forest call. If nothing changed, a real reason that points to your Phase 3 figures. "It was fine" gets 0 | 4 |
| **1** | Three new columns (a curve, a category, a combination) that only use their own row and never the target | 6 |
| | Each new column tied to a figure in one sentence | 2 |
| | `FEATURES` and `NEW_COLUMNS` run and everything is numeric | 2 |
| **2** | `past` and `future` made with the given code, and both shapes print | 3 |
| | Your answer on why unseen rows give a more honest score | 2 |
| **3** | Baseline computed from `past` with pandas (the mean for Track A, the most common class for Track B) and saved | 5 |
| | Baseline guess given in your target's units or class name, and its score explained in plain words | 5 |
| **4** | One feature picked, with a Phase 3 figure as the reason | 4 |
| | Right simple model for your track, fit on `past` and saved | 3 |
| | Compared to the baseline, in plain words with numbers | 3 |
| **5** | Model fit on `past` with all your features and saved | 3 |
| | Improvement over Part 4 given with numbers | 2 |
| | You used the comparison output to say whether your Part 1 columns helped | 2 |
| | You connected more features to a better score on learned data, and said what that means for trusting it | 3 |
| **6** | Three dial settings fit on `past` and saved (Track A says Ridge or Lasso, Track B uses three depths) | 6 |
| | Coefficient table shown (Track A) or all three depth scores shown (Track B) | 3 |
| | Questions 1 and 2 answered with specifics from your output | 4 |
| | Question 3 has a real guess with a reason, made before Part 8 | 2 |
| **7** | Random forest for your track fit on `past` and saved | 4 |
| | Feature importance shown and compared to your figures from Parts 1 and 4 | 3 |
| | Your answer on how much you trust the forest's `past` score | 3 |
| **8** | Results table shown | 2 |
| | Questions 1 to 3 answered with numbers from the table, including an overfitting explanation for the biggest change | 4 |
| | Verdict answers your research question with your best `future` score and names one thing the model can't tell you | 4 |
| **9** | README with your files and run order | 1 |
| | Contributions for each member | 1 |
| | AI use statement | 1 |
| | Runs top to bottom with Restart and Run All on `phase4_sample.csv`, and the last cell prints its message | 2 |
| | **Total** | **100** |

**Things that cost you points**

| What happened | Points off |
|---|---|
| Printing your whole DataFrame, or any output over 20 rows | 2 each time, up to 6 |
| Plotting, describing, or scoring `future` before Part 8 | 5 |
| Picking your Part 6 winner using `future` before Part 8 | 3 |
| A feature that leaks the target (built from it, or recorded after the outcome) | 5 |
| Notebook doesn't run top to bottom | The parts that don't run get 0 |
| Deleting parts or cells, or writing answers outside **Your answer** | 2 |